# SI 671 — 09 Final RQ3 / RQ4: Locked Test Evaluation and Heterogeneity

**RQ3.** Do historical OD-network features improve next-hour zone-level Yellow Taxi pickup forecasts beyond an enhanced temporal baseline?
**RQ4.** How do the resulting error changes vary across zones, demand levels, time, **RQ1 mobility communities** and **RQ2 temporal-demand clusters**?

This notebook is the **final, reportable** forecasting run. It supersedes the test numbers in `05_forecasting_revised.ipynb` (9-feature M1) and replaces the validation-only analyses in `07` and `08`:

| Item | 05_revised | 07 / 08 | **09 (this notebook)** |
|---|---|---|---|
| Temporal baseline | 9 features | 14 features (M1′) | 14 features (M1′, unchanged) |
| Boosting cap | 3,000 | 6,000 (all models hit the cap) | **20,000 + early stopping (200)** |
| Test evaluation | yes (weaker M1) | none | **once, M1′–M4′** |
| RQ4 groupings | demand, hour | demand, zone, hour, borough | **+ RQ1 community, RQ2 cluster**, val-vs-test consistency |

**Protocol (unchanged from 05/07).** Train Jan–Aug 2025; Validation Sep–Oct (early stopping only); Test Nov–Dec, scored once. One canonical sample (DST-affected history and incomplete network history excluded). Identical Poisson LightGBM settings across M1–M4, so each step isolates one feature block.

**Caveats to report.** (1) Nov–Dec was examined in earlier exploratory runs (05), so it is not a pristine holdout. (2) RQ1/RQ2 labels are full-year 2025 descriptive labels. They are used **only to group evaluation rows**, never as model features, so they cannot leak into forecasts. (3) Subgroup comparisons are exploratory, not multiplicity-adjusted tests. (4) Results describe Yellow Taxi pickups, not total NYC mobility (Uber/Lyft are discussed separately as an external-validity check).

## 0 — Configuration

Run from the repository root or `notebooks/` with the `si671-urban` kernel. RQ1/RQ2 tables come from Yiwu's branch (`results/network/...`, `results/temporal/...`); merge `origin/feature/temporal-clustering` first. Expected runtime on an M-series Mac: roughly 20–40 minutes, almost all LightGBM training.

In [ ]:
from pathlib import Path
import gc, json, platform, sys, warnings
from time import perf_counter

import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore", message=".*'eval_set' is deprecated.*")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.forecasting.evaluation import (
    add_errors, compare_subgroups, daily_block_bootstrap, direction_agreement,
    load_rq1_communities, load_rq2_clusters, overall_metrics, training_demand_groups,
)

CLEAN_DIR = PROJECT_ROOT / "data/processed/cleaned_trips"
ZONE_FILE = PROJECT_ROOT / "data/external/taxi_zone_lookup.csv"
AUG_FILE = PROJECT_ROOT / "data/processed/network_features/historical_network_features_24h_augmented.parquet"
OUT = PROJECT_ROOT / "results/forecasting/final_rq3_rq4"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

# RQ1 / RQ2 grouping labels (evaluation only, never model inputs).
# RQ1: change to a gamma=1.5 assignment file if the team adopts it as the main RQ1 result.
RQ1_COMMUNITY_FILE = PROJECT_ROOT / "results/network/tables/communities_annual.csv"
# RQ2: final min1000 K=3 clustering, used only for retrospective subgroup evaluation, not model inputs.
RQ2_DIR = PROJECT_ROOT / "results/temporal/sensitivity/min1000"
RQ2_CLUSTER_FILE = RQ2_DIR / "zone_temporal_clusters.csv"
RQ2_SUMMARY_FILE = RQ2_DIR / "cluster_summary.csv"
# cluster_id -> (expected peak hour in cluster_summary.csv, short name). Checked at load time.
RQ2_NAMES = {0: (7, "Commuter (weekday AM peak)"),
             1: (23, "Core activity (afternoon-evening)"),
             2: (0, "Nightlife (weekend late night)")}

SMOKE_TEST = False          # True = quick pipeline check only; never report its numbers
MAX_ROUNDS = 300 if SMOKE_TEST else 20_000
EARLY_STOPPING_ROUNDS = 30 if SMOKE_TEST else 200
LEARNING_RATE = 0.05        # same as 07; only the cap changes, so results stay comparable
NUM_THREADS = 4
RANDOM_SEED = 671
N_BOOT = 200 if SMOKE_TEST else 2000
TRAIN_END = pd.Timestamp("2025-09-01")
VAL_END = pd.Timestamp("2025-11-01")
START, END = pd.Timestamp("2025-01-01"), pd.Timestamp("2026-01-01")
TARGET = "pickup_count"

for path in [ZONE_FILE, AUG_FILE, RQ1_COMMUNITY_FILE, RQ2_CLUSTER_FILE, RQ2_SUMMARY_FILE]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required input: {path}")
monthly_paths = [CLEAN_DIR / f"cleaned_2025-{m:02d}.parquet" for m in range(1, 13)]
assert all(p.is_file() for p in monthly_paths), "All 12 cleaned_2025-MM.parquet files are required"
print("Python/pandas/LightGBM:", platform.python_version(), pd.__version__, lgb.__version__)
print("Run type:", "SMOKE TEST (not reportable)" if SMOKE_TEST else "FINAL RUN")
print("Boosting cap:", MAX_ROUNDS, "| early stopping:", EARLY_STOPPING_ROUNDS, "| lr:", LEARNING_RATE)

## 1 — 2025 zone-hour pickup panel (identical to notebooks 05–07)

In [ ]:
zone_lookup = pd.read_csv(ZONE_FILE)
forecast_zone_ids = sorted(set(zone_lookup["LocationID"].dropna().astype(int)) - {264, 265})
assert len(forecast_zone_ids) == 263
monthly_panels, source_report = [], []
for month, path in enumerate(monthly_paths, start=1):
    trips = pd.read_parquet(path, columns=["tpep_pickup_datetime", "PULocationID"])
    month_start = pd.Timestamp(2025, month, 1)
    month_end = month_start + pd.DateOffset(months=1)
    trips = trips.loc[trips["PULocationID"].isin(forecast_zone_ids)].copy()
    times = pd.to_datetime(trips["tpep_pickup_datetime"])
    if ((times < month_start) | (times >= month_end) | times.isna()).any():
        raise ValueError(f"{path.name}: pickup timestamps outside month")
    trips["pickup_hour"] = times.dt.floor("h")
    grouped = trips.groupby(["PULocationID", "pickup_hour"], observed=True).size()
    hours = pd.date_range(month_start, month_end, freq="h", inclusive="left")
    keys = pd.MultiIndex.from_product([forecast_zone_ids, hours], names=["zone_id", "pickup_hour"])
    panel = grouped.rename_axis(["zone_id", "pickup_hour"]).reindex(keys, fill_value=0).rename(TARGET).reset_index()
    panel[TARGET] = panel[TARGET].astype("int32")
    assert panel[TARGET].sum() == len(trips)
    monthly_panels.append(panel)
    source_report.append({"month": month_start.strftime("%Y-%m"), "geographic_pickups": len(trips)})
    del trips, panel, grouped, keys

full_year_panel = (pd.concat(monthly_panels, ignore_index=True)
                   .sort_values(["zone_id", "pickup_hour"]).reset_index(drop=True))
del monthly_panels
n_hours = len(pd.date_range(START, END, freq="h", inclusive="left"))
assert len(full_year_panel) == 263 * n_hours
assert not full_year_panel.duplicated(["zone_id", "pickup_hour"]).any()
print(f"Panel: {len(full_year_panel):,} zone-hours")

## 2 — Time-safe temporal features and DST exclusion (identical to 07)

In [ ]:
forecast_df = full_year_panel.copy()
series = forecast_df.groupby("zone_id", sort=False, observed=True)[TARGET]
for lag in [1, 2, 3, 24, 168]:
    forecast_df[f"lag_{lag}"] = series.shift(lag).astype("float32")
for window in [3, 6, 24, 168]:
    forecast_df[f"rolling_mean_{window}"] = series.transform(
        lambda x: x.shift(1).rolling(window, min_periods=window).mean()).astype("float32")
forecast_df["trend_3_vs_24"] = (forecast_df["rolling_mean_3"] - forecast_df["rolling_mean_24"]).astype("float32")
t = forecast_df["pickup_hour"]
forecast_df["hour"] = t.dt.hour.astype("int8")
forecast_df["day_of_week"] = t.dt.day_of_week.astype("int8")
forecast_df["is_weekend"] = (forecast_df["day_of_week"] >= 5).astype("int8")

DST_BAD_HOURS = [pd.Timestamp("2025-03-09 02:00:00"), pd.Timestamp("2025-11-02 01:00:00")]
forecast_df["dst_history_affected"] = False
for h in DST_BAD_HOURS:
    forecast_df.loc[forecast_df["pickup_hour"].between(h, h + pd.Timedelta(hours=168)),
                    "dst_history_affected"] = True

ORIGINAL_TEMPORAL = ["lag_1", "lag_24", "lag_168", "rolling_mean_24", "rolling_mean_168",
                     "hour", "day_of_week", "is_weekend", "zone_id"]
SHORT_TERM = ["lag_2", "lag_3", "rolling_mean_3", "rolling_mean_6", "trend_3_vs_24"]
TEMPORAL = ORIGINAL_TEMPORAL + SHORT_TERM

for zone, dt in [(161, "2025-06-10 15:00"), (43, "2025-12-15 10:00")]:
    hit = forecast_df.loc[(forecast_df.zone_id == zone) & (forecast_df.pickup_hour == pd.Timestamp(dt))].iloc[0]
    past = full_year_panel.loc[(full_year_panel.zone_id == zone) &
                               (full_year_panel.pickup_hour < pd.Timestamp(dt))].sort_values("pickup_hour")
    assert np.isclose(hit["lag_1"], past[TARGET].iloc[-1])
    assert np.isclose(hit["rolling_mean_24"], past[TARGET].iloc[-24:].mean())

model_df = (forecast_df.loc[~forecast_df["dst_history_affected"]]
            .dropna(subset=TEMPORAL + [TARGET])
            .sort_values(["pickup_hour", "zone_id"]).reset_index(drop=True))
assert len(model_df) == 2_170_802, f"Unexpected modeling rows: {len(model_df):,}"
print(f"Modeling rows: {len(model_df):,}")

## 3 — Historical OD-network features: canonical merge (identical to 07)

In [ ]:
STRENGTH = ["in_strength_24h", "out_strength_24h", "interzone_in_strength_24h", "interzone_out_strength_24h"]
DEGREE = ["in_degree_24h", "out_degree_24h"]
HHI = ["in_hhi_24h", "out_hhi_24h"]
FLOW_FLAGS = ["has_interzone_in_flow_24h", "has_interzone_out_flow_24h"]
net = pd.read_parquet(AUG_FILE)
net["zone_id"] = net["zone_id"].astype("int32")
net["feature_as_of"] = pd.to_datetime(net["feature_as_of"])
assert not net.duplicated(["zone_id", "feature_as_of"]).any()
model_df["zone_id"] = model_df["zone_id"].astype("int32")
merged = model_df.merge(net, how="left", left_on=["zone_id", "pickup_hour"],
                        right_on=["zone_id", "feature_as_of"], validate="one_to_one", indicator=True)
assert merged["_merge"].eq("both").all()
valid = (merged["has_full_history_24h"].fillna(False).astype(bool)
         & merged["dst_safe_24h"].fillna(False).astype(bool))
canonical_df = merged.loc[valid].drop(columns=["_merge"]).sort_values(["pickup_hour", "zone_id"]).reset_index(drop=True)
del merged, net, forecast_df
gc.collect()
assert canonical_df[STRENGTH + DEGREE].notna().all().all()
for d in ["in", "out"]:
    assert (canonical_df[f"{d}_hhi_24h"].isna() == ~canonical_df[f"has_interzone_{d}_flow_24h"].astype(bool)).all()
print(f"Canonical sample: {len(canonical_df):,}")

## 4 — Chronological split (Train / Validation / Test)

In [ ]:
split = np.select([canonical_df["pickup_hour"] < TRAIN_END, canonical_df["pickup_hour"] < VAL_END],
                  ["Train", "Validation"], default="Test")
train = canonical_df.loc[split == "Train"].copy()
val = canonical_df.loc[split == "Validation"].copy()
test = canonical_df.loc[split == "Test"].copy()
assert [len(train), len(val), len(test)] == [1_445_185, 385_032, 340_585], (len(train), len(val), len(test))
assert train.pickup_hour.max() < val.pickup_hour.min() and val.pickup_hour.max() < test.pickup_hour.min()
zone_dtype = pd.CategoricalDtype(categories=forecast_zone_ids)
for d in (train, val, test):
    d["zone_id"] = d["zone_id"].astype(zone_dtype)
print({"Train": len(train), "Validation": len(val), "Test": len(test)})

## 5 — Feature sets and shared model configuration

M1′ temporal → M2′ + OD strength → M3′ + degree → M4′ + HHI and flow flags. Only the boosting cap differs from 07; every model gets the same budget and stopping rule.

In [ ]:
FEATURE_SETS = {
    "m1": TEMPORAL,
    "m2": TEMPORAL + STRENGTH,
    "m3": TEMPORAL + STRENGTH + DEGREE,
    "m4": TEMPORAL + STRENGTH + DEGREE + HHI + FLOW_FLAGS,
}
MODEL_NAMES = {"m1": "M1 Temporal", "m2": "M2 + OD Strength", "m3": "M3 + Degree",
               "m4": "M4 + HHI & Flow Flags", "b0d": "B0 Daily Naive (lag 24)",
               "b0w": "B0 Weekly Naive (lag 168)"}
PARAMS = dict(objective="poisson", metric="l1", n_estimators=MAX_ROUNDS, learning_rate=LEARNING_RATE,
              num_leaves=31, min_child_samples=20, random_state=RANDOM_SEED, n_jobs=NUM_THREADS, verbosity=-1)
for code, cols in FEATURE_SETS.items():
    assert len(cols) == len(set(cols)) and set(cols).issubset(train.columns)
print(pd.DataFrame([{"model": MODEL_NAMES[c], "n_features": len(f)} for c, f in FEATURE_SETS.items()]))

## 6 — Train M1′–M4′ (early stopping on Validation), predict Validation and Test once

In [ ]:
key_cols = ["zone_id", "pickup_hour", TARGET]
pred_val = val[key_cols].copy().reset_index(drop=True)
pred_test = test[key_cols].copy().reset_index(drop=True)
for frame, src in [(pred_val, val), (pred_test, test)]:
    frame["pred_b0d"] = src["lag_24"].to_numpy(dtype=float)
    frame["pred_b0w"] = src["lag_168"].to_numpy(dtype=float)

fit_rows, curves, importances = [], {}, []
for i, (code, cols) in enumerate(FEATURE_SETS.items(), start=1):
    print(f"[{i}/4] {MODEL_NAMES[code]} ({len(cols)} features)", flush=True)
    t0 = perf_counter()
    history = {}
    model = lgb.LGBMRegressor(**PARAMS)
    model.fit(train[cols], train[TARGET], eval_set=[(val[cols], val[TARGET])], eval_metric="l1",
              categorical_feature=["zone_id"],
              callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, first_metric_only=True, verbose=False),
                         lgb.record_evaluation(history), lgb.log_evaluation(2000)])
    best = int(model.best_iteration_ or MAX_ROUNDS)
    for frame, src in [(pred_val, val), (pred_test, test)]:
        p = np.asarray(model.predict(src[cols], num_iteration=best), dtype=float)
        assert np.isfinite(p).all() and (p >= 0).all()
        frame[f"pred_{code}"] = p
    fit_rows.append({"code": code, "model": MODEL_NAMES[code], "n_features": len(cols), "best_iteration": best,
                     "hit_round_cap": best >= MAX_ROUNDS - EARLY_STOPPING_ROUNDS,
                     "train_seconds": round(perf_counter() - t0, 1)})
    curves[code] = [float(x) for x in history["valid_0"]["l1"]]
    importances.append(pd.DataFrame({"code": code, "feature": cols,
                                     "gain": model.booster_.feature_importance("gain"),
                                     "split": model.booster_.feature_importance("split")}))
    print(f"    best_iteration={best} | {fit_rows[-1]['train_seconds']}s", flush=True)
    del model
    gc.collect()

fit_df = pd.DataFrame(fit_rows)
importance_df = pd.concat(importances, ignore_index=True)
display(fit_df)
if fit_df["hit_round_cap"].any():
    warnings.warn("A model still reached the boosting cap: raise MAX_ROUNDS or LEARNING_RATE before reporting.")

## 7 — RQ3: locked Test results

Primary metric MAE, secondary RMSE. `*_reduction_vs_prev` is the incremental effect of the newest feature block; `*_vs_first` is the cumulative change relative to M1′. Validation rows are shown for transparency (they were used for early stopping).

In [ ]:
CODES = ["m1", "m2", "m3", "m4"]
for frame in (pred_val, pred_test):
    add_errors(frame, CODES + ["b0d", "b0w"])
rq3 = pd.concat([overall_metrics(pred_val, CODES, MODEL_NAMES, "Validation"),
                 overall_metrics(pred_test, CODES, MODEL_NAMES, "Test")], ignore_index=True)
rq3 = rq3.merge(fit_df[["code", "best_iteration", "hit_round_cap"]], on="code", how="left")
baselines = pd.concat([overall_metrics(f, ["b0d", "b0w"], MODEL_NAMES, s)[["split", "model", "n_obs", "MAE", "RMSE"]]
                       for f, s in [(pred_val, "Validation"), (pred_test, "Test")]], ignore_index=True)
display(rq3[["split", "model", "MAE", "RMSE", "MAE_reduction_vs_prev", "MAE_reduction_vs_prev_pct",
             "MAE_reduction_vs_first_pct", "RMSE_reduction_vs_prev_pct", "best_iteration"]].round(5))
display(baselines.round(4))

PAIRS = [("m1", "m2"), ("m2", "m3"), ("m3", "m4"), ("m1", "m4")]
rq3_boot = pd.concat([daily_block_bootstrap(pred_test, o, n, n_boot=N_BOOT, seed=RANDOM_SEED) for o, n in PAIRS],
                     ignore_index=True)
display(rq3_boot.round(5))

pred_test["month"] = pred_test["pickup_hour"].dt.strftime("%Y-%m")
rq3_month = pd.concat([compare_subgroups(pred_test, "month", o, n) for o, n in PAIRS[:3]], ignore_index=True)
display(rq3_month[["comparison", "month", "n", "old_MAE", "new_MAE", "abs_MAE_reduction",
                   "relative_MAE_reduction_pct"]].round(5))

## 8 — RQ4 grouping labels

* **Demand level:** equal-sized tertiles of Jan–Aug pickups per hour (training period only).
* **Borough:** TLC lookup.
* **RQ1 community:** Louvain community of the full-year OD network.
* **RQ2 cluster:** min1000 weekday-hour temporal cluster; zones under 1,000 annual pickups are `Unclustered (low volume)`.
* **Time:** hour of day, weekday/weekend.

In [ ]:
train_counts = (full_year_panel.loc[full_year_panel["pickup_hour"] < TRAIN_END]
                .groupby("zone_id")[TARGET].sum().reindex(forecast_zone_ids, fill_value=0))
n_train_hours = len(pd.date_range(START, TRAIN_END, freq="h", inclusive="left"))
demand = training_demand_groups(train_counts, n_train_hours)
labels = (demand[["zone_id", "demand_group", "train_mean_pickups_per_hour"]]
          .merge(zone_lookup.rename(columns={"LocationID": "zone_id"})[["zone_id", "Borough", "Zone"]],
                 on="zone_id", how="left", validate="one_to_one")
          .merge(load_rq1_communities(RQ1_COMMUNITY_FILE), on="zone_id", how="left", validate="one_to_one")
          .merge(load_rq2_clusters(RQ2_CLUSTER_FILE, RQ2_SUMMARY_FILE, RQ2_NAMES),
                 on="zone_id", how="left", validate="one_to_one"))
labels["rq1_community"] = labels["rq1_community"].fillna("No OD trips")
labels["rq2_cluster"] = labels["rq2_cluster"].fillna("Unclustered (low volume)")
assert len(labels) == 263 and labels["demand_group"].notna().all()

def attach_labels(frame):
    out = frame.assign(zone_id=frame["zone_id"].astype(int)).merge(labels, on="zone_id", how="left",
                                                                     validate="many_to_one")
    assert len(out) == len(frame) and out["demand_group"].notna().all()
    out["hour"] = out["pickup_hour"].dt.hour
    out["day_type"] = np.where(out["pickup_hour"].dt.day_of_week >= 5, "Weekend", "Weekday")
    return out

pred_val, pred_test = attach_labels(pred_val), attach_labels(pred_test)

display(pd.crosstab(labels["rq2_cluster"], labels["demand_group"], margins=True))
display(pd.crosstab(labels["rq1_community"], labels["rq2_cluster"], margins=True))

## 9 — RQ4: where and when does OD strength help? (M1′ → M2′, Test)

Each table is observation-weighted. `share_of_net_gain_pct` > 100% for a group means other groups lose accuracy. The val-vs-test table checks whether each subgroup's direction of change replicates across periods; replicated signs are more credible than single-period magnitudes.

In [ ]:
GROUPINGS = ["demand_group", "rq1_community", "rq2_cluster", "Borough", "day_type", "hour"]
rq4_test = {g: compare_subgroups(pred_test, g) for g in GROUPINGS}
rq4_val = {g: compare_subgroups(pred_val, g) for g in GROUPINGS}
show = ["n_zones", "n", "mean_actual_pickups", "old_MAE", "new_MAE", "abs_MAE_reduction",
        "relative_MAE_reduction_pct", "RMSE_reduction", "share_of_net_gain_pct"]
for g in ["demand_group", "rq1_community", "rq2_cluster", "Borough", "day_type"]:
    print(f"\n=== Test, by {g} ===")
    display(rq4_test[g][[g] + show].round(4))

consistency = pd.concat([direction_agreement(rq4_val[g], rq4_test[g], g).rename(columns={g: "group"}).assign(grouping=g)
                         for g in GROUPINGS], ignore_index=True)
print("\nSubgroups whose M1->M2 gain has the same sign in Validation and Test:")
display(consistency.groupby("grouping")["same_sign"].agg(["sum", "count"]))

rq4_boot = pd.concat([daily_block_bootstrap(pred_test, "m1", "m2", n_boot=N_BOOT, seed=RANDOM_SEED, group_col=g)
                      for g in ["demand_group", "rq1_community", "rq2_cluster"]], ignore_index=True)
rq4_boot = rq4_boot.loc[~((rq4_boot["group"] == "ALL") & rq4_boot.duplicated("group"))]
display(rq4_boot.round(5))

# Zone level, and the interaction most relevant to RQ2: cluster x hour of day
zone_test = compare_subgroups(pred_test, "zone_id").merge(labels, on="zone_id", how="left")
zone_val = compare_subgroups(pred_val, "zone_id")[["zone_id", "abs_MAE_reduction"]]
zone_test = zone_test.merge(zone_val.rename(columns={"abs_MAE_reduction": "abs_MAE_reduction_val"}), on="zone_id")
print("Zones improved on Test:", int((zone_test.abs_MAE_reduction > 0).sum()), "of", len(zone_test))
print("Zones improved in both Validation and Test:",
      int(((zone_test.abs_MAE_reduction > 0) & (zone_test.abs_MAE_reduction_val > 0)).sum()))
print("Zone-level gain correlation, Validation vs Test (Spearman):",
      round(zone_test[["abs_MAE_reduction", "abs_MAE_reduction_val"]].corr("spearman").iloc[0, 1], 3))
display(zone_test.nlargest(10, "abs_MAE_reduction")[["zone_id", "Zone", "Borough", "rq2_cluster", "demand_group",
                                                      "old_MAE", "abs_MAE_reduction", "abs_MAE_reduction_val"]].round(4))
cluster_hour = compare_subgroups(pred_test, ["rq2_cluster", "hour"])

# Secondary blocks by RQ2 cluster and demand: do degree / HHI help anywhere?
secondary = pd.concat([compare_subgroups(pred_test, g, o, n).rename(columns={g: "group"}).assign(grouping=g)
                       for o, n in [("m2", "m3"), ("m3", "m4")] for g in ["demand_group", "rq2_cluster"]],
                      ignore_index=True)
display(secondary[["comparison", "grouping", "group", "n", "abs_MAE_reduction", "relative_MAE_reduction_pct"]].round(5))

## 10 — Figures

In [ ]:
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# Fig 1 — RQ3: incremental Test gain of each feature block with daily-block 95% intervals
fig, ax = plt.subplots(figsize=(7.5, 4))
b = rq3_boot.loc[rq3_boot["group"] == "ALL"].reset_index(drop=True)
y = b["observed_abs_MAE_reduction"]
ax.errorbar(range(len(b)), y, yerr=[y - b["ci_lower_2.5pct"], b["ci_upper_97.5pct"] - y],
            fmt="o", capsize=5, color="#1f4e79")
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(range(len(b)), ["+ Strength\n(M1→M2)", "+ Degree\n(M2→M3)", "+ HHI/flags\n(M3→M4)", "All network\n(M1→M4)"])
ax.set_ylabel("MAE reduction (pickups / zone-hour)\npositive = network features help")
ax.set_title("RQ3 — Test-period gain from each network feature block")
fig.tight_layout(); fig.savefig(FIG / "rq3_test_block_gains.png", dpi=180); plt.show()

# Fig 2 — learning curves (convergence check)
fig, ax = plt.subplots(figsize=(7.5, 4))
for code, c in curves.items():
    ax.plot(np.arange(1, len(c) + 1), c, label=MODEL_NAMES[code])
ax.set_xlabel("Boosting iteration"); ax.set_ylabel("Validation MAE"); ax.legend()
ax.set_ylim(top=np.percentile(np.concatenate([np.array(c) for c in curves.values()]), 90))
ax.set_title("Validation learning curves (early stopping at 200 rounds without improvement)")
fig.tight_layout(); fig.savefig(FIG / "learning_curves.png", dpi=180); plt.show()

# Fig 3 — RQ4: M1→M2 gain with intervals by demand, RQ1 community, RQ2 cluster
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharey=True)
for ax, g, title in zip(axes, ["demand_group", "rq1_community", "rq2_cluster"],
                        ["Training demand level", "RQ1 mobility community", "RQ2 temporal cluster"]):
    d = rq4_boot.loc[(rq4_boot["grouping"] == g) & (rq4_boot["group"] != "ALL")].reset_index(drop=True)
    y = d["observed_abs_MAE_reduction"]
    ax.errorbar(range(len(d)), y, yerr=[y - d["ci_lower_2.5pct"], d["ci_upper_97.5pct"] - y],
                fmt="o", capsize=5, color="#1f4e79")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(range(len(d)), [s.replace(" (", "\n(") for s in d["group"]], fontsize=8)
    ax.set_title(title)
axes[0].set_ylabel("M1′ MAE − M2′ MAE (Test)")
fig.suptitle("RQ4 — Where historical OD strength helps (daily-block 95% intervals)")
fig.tight_layout(); fig.savefig(FIG / "rq4_gain_by_group.png", dpi=180); plt.show()

# Fig 4 — RQ4: hourly gain by RQ2 cluster
fig, ax = plt.subplots(figsize=(9, 4.5))
for name, d in cluster_hour.groupby("rq2_cluster"):
    ax.plot(d["hour"], d["abs_MAE_reduction"], marker="o", ms=3, label=name)
ax.axhline(0, color="black", lw=0.8); ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("Hour of day"); ax.set_ylabel("M1′ MAE − M2′ MAE (Test)")
ax.set_title("RQ4 — Hourly gain from OD strength, by RQ2 temporal cluster"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "rq4_hourly_gain_by_rq2_cluster.png", dpi=180); plt.show()

# Fig 5 — RQ4: zone gains, Validation vs Test (replication)
fig, ax = plt.subplots(figsize=(6, 5.5))
colors = zone_test["demand_group"].astype(str).map({"Low": "#bbbbbb", "Medium": "#7aa6c2", "High": "#1f4e79"})
ax.scatter(zone_test["abs_MAE_reduction_val"], zone_test["abs_MAE_reduction"], c=colors, s=14)
ax.axhline(0, color="black", lw=0.6); ax.axvline(0, color="black", lw=0.6)
for _, r in zone_test.nlargest(5, "abs_MAE_reduction").iterrows():
    ax.annotate(str(r["Zone"]), (r["abs_MAE_reduction_val"], r["abs_MAE_reduction"]), fontsize=7)
ax.set_xlabel("Zone gain, Validation"); ax.set_ylabel("Zone gain, Test")
ax.set_title("Zone-level M1′→M2′ gain replicates? (dark = high demand)")
fig.tight_layout(); fig.savefig(FIG / "rq4_zone_gain_val_vs_test.png", dpi=180); plt.show()

## 11 — Save artifacts and a summary for the report

CSV/PNG/JSON are small and should be committed. Prediction parquet files are git-ignored (`*.parquet`).

In [ ]:
rq3.to_csv(OUT / "rq3_metrics.csv", index=False)
baselines.to_csv(OUT / "rq3_baselines.csv", index=False)
rq3_boot.to_csv(OUT / "rq3_test_bootstrap.csv", index=False)
rq3_month.to_csv(OUT / "rq3_test_by_month.csv", index=False)
fit_df.to_csv(OUT / "model_fit_summary.csv", index=False)
importance_df.to_csv(OUT / "feature_importance.csv", index=False)
for g in GROUPINGS:
    rq4_test[g].to_csv(OUT / f"rq4_test_by_{g}.csv", index=False)
    rq4_val[g].to_csv(OUT / f"rq4_validation_by_{g}.csv", index=False)
consistency.to_csv(OUT / "rq4_val_test_consistency.csv", index=False)
rq4_boot.to_csv(OUT / "rq4_test_bootstrap_by_group.csv", index=False)
zone_test.to_csv(OUT / "rq4_test_by_zone.csv", index=False)
cluster_hour.to_csv(OUT / "rq4_test_rq2_cluster_by_hour.csv", index=False)
secondary.to_csv(OUT / "rq4_test_secondary_blocks.csv", index=False)
labels.to_csv(OUT / "rq4_zone_labels.csv", index=False)
pred_cols = ["zone_id", "pickup_hour", TARGET] + [f"pred_{c}" for c in CODES + ["b0d", "b0w"]]
pred_val[pred_cols].to_parquet(OUT / "validation_predictions.parquet", index=False)
pred_test[pred_cols].to_parquet(OUT / "test_predictions.parquet", index=False)
(OUT / "learning_curves.json").write_text(json.dumps(curves))
manifest = {
    "notebook": "09_final_rq3_rq4.ipynb", "smoke_test": SMOKE_TEST, "params": PARAMS,
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS, "feature_sets": FEATURE_SETS,
    "splits": {"train": len(train), "validation": len(val), "test": len(test),
               "train_end_exclusive": str(TRAIN_END), "validation_end_exclusive": str(VAL_END)},
    "rq1_file": str(RQ1_COMMUNITY_FILE.relative_to(PROJECT_ROOT)),
    "rq2_file": str(RQ2_CLUSTER_FILE.relative_to(PROJECT_ROOT)), "rq2_names": RQ2_NAMES,
    "best_iterations": dict(zip(fit_df.code, fit_df.best_iteration.astype(int))),
    "versions": {"python": platform.python_version(), "pandas": pd.__version__, "lightgbm": lgb.__version__},
}
(OUT / "experiment_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))

# Plain-text key numbers to paste into the report (generated, never hand-typed)
res = rq3.loc[rq3.split == "Test"].set_index("code")
naive_test = baselines.loc[baselines.split == "Test"].reset_index(drop=True)
bb = rq3_boot.loc[rq3_boot.group == "ALL"].set_index("comparison")
dg = rq4_test["demand_group"].set_index("demand_group")
lines = [
    "# Final RQ3/RQ4 key numbers (Test, Nov–Dec 2025)", "",
    f"- M1′ MAE {res.loc['m1','MAE']:.4f}, RMSE {res.loc['m1','RMSE']:.4f}; "
    f"M2′ MAE {res.loc['m2','MAE']:.4f}, RMSE {res.loc['m2','RMSE']:.4f}",
    f"- M1′→M2′ MAE reduction {res.loc['m2','MAE_reduction_vs_prev']:.4f} ({res.loc['m2','MAE_reduction_vs_prev_pct']:.2f}%), "
    f"95% daily-block CI [{bb.loc['M1->M2','ci_lower_2.5pct']:.4f}, {bb.loc['M1->M2','ci_upper_97.5pct']:.4f}]; "
    f"RMSE reduction {res.loc['m2','RMSE_reduction_vs_prev_pct']:.2f}%",
    f"- M2′→M3′ (degree) {res.loc['m3','MAE_reduction_vs_prev']:+.4f}, CI [{bb.loc['M2->M3','ci_lower_2.5pct']:.4f}, {bb.loc['M2->M3','ci_upper_97.5pct']:.4f}]",
    f"- M3′→M4′ (HHI/flags) {res.loc['m4','MAE_reduction_vs_prev']:+.4f}, CI [{bb.loc['M3->M4','ci_lower_2.5pct']:.4f}, {bb.loc['M3->M4','ci_upper_97.5pct']:.4f}]",
    f"- Best iterations: {manifest['best_iterations']}; any model at cap: {bool(fit_df.hit_round_cap.any())}",
    f"- Seasonal naive Test MAE: daily {naive_test.MAE.iloc[0]:.4f}, weekly {naive_test.MAE.iloc[1]:.4f}",
    f"- Zones improved (Test): {int((zone_test.abs_MAE_reduction > 0).sum())}/263; in both periods: "
    f"{int(((zone_test.abs_MAE_reduction > 0) & (zone_test.abs_MAE_reduction_val > 0)).sum())}",
    "- Gain by demand group: " + "; ".join(f"{k} {dg.loc[k,'abs_MAE_reduction']:+.4f} ({dg.loc[k,'share_of_net_gain_pct']:.0f}% of net)"
                                           for k in ["Low", "Medium", "High"]),
    "- Gain by RQ2 cluster: " + "; ".join(f"{r.rq2_cluster} {r.abs_MAE_reduction:+.4f}" for r in rq4_test["rq2_cluster"].itertuples()),
    "- Gain by RQ1 community: " + "; ".join(f"{r.rq1_community} {r.abs_MAE_reduction:+.4f}" for r in rq4_test["rq1_community"].itertuples()),
]
(OUT / "summary_for_report.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))
print("\nSaved to", OUT.relative_to(PROJECT_ROOT))

## 12 — How to read the results (fill in after the run)

Write the findings **from `summary_for_report.md` and the tables above**, not from earlier notebooks.

**RQ3.** Report the Test M1′→M2′ MAE and RMSE change with its daily-block interval. If the interval excludes zero, say OD strength gives a small, stable improvement; describe its size relative to the temporal baseline's gain over the seasonal-naive baselines, which is far larger. Report degree and HHI as separate blocks; a negative or null effect is a valid finding (degree is highly collinear with strength and demand). State whether any model reached the boosting cap.

**RQ4.** Lead with demand level: report where the net gain concentrates and whether low/medium-demand zones lose accuracy. Then RQ1 communities and RQ2 clusters: does the gain concentrate in one community or temporal type (for example the Manhattan core or nightlife zones)? Use the Fig. 4 curves to say *when* in the day the gain appears for each cluster. Use the val-vs-test consistency table to separate replicated patterns from single-period noise.

**Limitations.** Nov–Dec previously inspected; one train/validation/test split; subgroup tables are exploratory and not multiplicity-adjusted; RQ1/RQ2 labels are full-year descriptive groupings used only for evaluation; Yellow Taxi only; offline assumption that completed-trip records are available by the forecast hour; DST-affected hours excluded.